Tensor Flow

In [1]:
import tensorflow as tf

In [3]:
# 1. 필요한 라이브러리 임포트
# TensorFlow와 Keras의 기능을 사용하기 위해 필요한 모듈을 불러옵니다.
import numpy as np
from tensorflow.keras.models import Sequential              # 순차적 모델을 생성하기 위한 모듈
from tensorflow.keras.layers import Dense, Input,Dropout    # 밀집층(fully connected layer)을 추가하기 위한 모듈
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split        # 데이터를 학습/테스트 세트로 나누기 위한 모듈
from sklearn.datasets import make_classification            # 예제 데이터셋 생성 모듈


In [4]:
# 데이터 생성
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_classes=2,
    random_state=42
)

In [7]:
# 데이터 분리 8:2
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# 모델 생성 : 레이어 층 구성
model = Sequential([
    Input(shape=(X_train.shape[1], )),      # 입력층, 열이 들어가야해서 [1]

    Dense(8, activation="relu"),            # 은닉층 : 핵심 레이어 -> 패턴 찾기, 데이터의 복잡성에 따라서 은닉층을 여러 개 배치 가능
    
    Dense(1, activation="sigmoid")          # 출력층
])

In [10]:
# 학습 방법 정의 : 옵티마이저, 손실함수, 평가지표
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [14]:
history = model.fit(
    X_train, y_train,
    #validation_data=()
    validation_split=0.2,
    epochs=50,
    batch_size=32
)

Epoch 1/50


20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8922 - loss: 0.3027 - val_accuracy: 0.8625 - val_loss: 0.2879
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8922 - loss: 0.3014 - val_accuracy: 0.8625 - val_loss: 0.2859
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8953 - loss: 0.2998 - val_accuracy: 0.8625 - val_loss: 0.2854
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8984 - loss: 0.2983 - val_accuracy: 0.8625 - val_loss: 0.2846
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8969 - loss: 0.2968 - val_accuracy: 0.8625 - val_loss: 0.2824
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8984 - loss: 0.2957 - val_accuracy: 0.8625 - val_loss: 0.2809
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8969 - loss: 0.2946 - val_accuracy: 0.8562 - val_loss: 0.2799
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8984 - loss: 0.2932 - val_accuracy: 0.8562 - val_loss: 0.2794
Epo

In [15]:
# 평가 지표
test_loss, test_acc = model.evaluate(X_test, y_test)

print(test_loss)
print(test_acc)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8550 - loss: 0.3730 
0.37299850583076477
0.8550000190734863


In [16]:
# 예측
predictions = model.predict(X_test[:5])

print(predictions)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
[[0.6521554 ]
 [0.6402497 ]
 [0.20678148]
 [0.94299126]
 [0.9561429 ]]


과적합 예방

In [17]:
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_classes=2,
    random_state=42
)

In [18]:
# 7:3
#   => 3 -> 5:5
# 최종적으로 70:15:15
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)   # 70:30

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)   # 15:15

In [19]:
from sklearn.preprocessing import StandardScaler

In [20]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [23]:
# 모델 구성
model = Sequential([
    Input(shape=(X_train.shape[1], )),       # 입력층

    Dense(64, activation="relu"),            # 첫 번째 은닉층

    # 과적합 방지
    Dropout(0.5),

    Dense(32, activation="relu"),            # 두 번째 은닉층

    # Dense(1, activation=None)              # 수치 예측, 양수만 나와야 한다면 relu
    Dense(1, activation="sigmoid")           # 출력층 : 이진 분류
])

In [24]:
# 학습 방법 정의 : 옵티마이저, 손실함수, 평가지표
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [26]:
# 과적합 방지
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

# 기능정의 => val_loss를 보며 손실이 적어지면 멈추겠다 

In [27]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    # early_stopping 설정
    callbacks=[early_stopping]
)

Epoch 1/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5957 - loss: 0.6797 - val_accuracy: 0.7467 - val_loss: 0.5665
Epoch 2/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7329 - loss: 0.5675 - val_accuracy: 0.7800 - val_loss: 0.4997
Epoch 3/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7657 - loss: 0.5220 - val_accuracy: 0.7867 - val_loss: 0.4527
Epoch 4/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8043 - loss: 0.4589 - val_accuracy: 0.8067 - val_loss: 0.4153
Epoch 5/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8100 - loss: 0.4273 - val_accuracy: 0.8067 - val_loss: 0.3917
Epoch 6/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8386 - loss: 0.3928 - val_accuracy: 0.8067 - val_loss: 0.3768
Epoch 7/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8414 - loss: 0.3909 - val_accuracy: 0.8267 - val_loss: 0.3621
Epoch 8/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8586 - loss: 0.3664 - val_accuracy: 0.8467 -

In [29]:
test_loss, test_acc = model.evaluate(X_test, y_test)

print(test_acc)
print(test_loss)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8400 - loss: 0.4525 
0.8399999737739563
0.4524509608745575


다중분류

In [30]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import mnist

In [31]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [32]:
# 입력 데이터 : 이미지 데이터 => 1차원 벡터 변환 (정규화)
X_train = X_train.reshape(-1, 784) / 255.0          # 28 * 28 = 784
X_test = X_test.reshape(-1, 784) / 255.0

In [33]:
# 라벨 => 원핫인코딩
y_train = to_categorical(y_train, num_classes=10)
y_test = to_categorical(y_test, num_classes=10)

In [35]:
model = Sequential([
    Input(shape=(784,)),

    Dense(64, activation="relu"),
    
    Dense(10, activation="softmax")       # 출력층
])

In [36]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [37]:
model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9079 - loss: 0.3325 - val_accuracy: 0.9413 - val_loss: 0.2015
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9512 - loss: 0.1642 - val_accuracy: 0.9575 - val_loss: 0.1477
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9651 - loss: 0.1190 - val_accuracy: 0.9638 - val_loss: 0.1264
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9722 - loss: 0.0946 - val_accuracy: 0.9657 - val_loss: 0.1158
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9765 - loss: 0.0776 - val_accuracy: 0.9676 - val_loss: 0.1121
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9808 - loss: 0.0634 - val_accuracy: 0.9677 - val_loss: 0.1094
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9835 - loss: 0.0541 - val_accuracy: 0.9672 - val_loss: 0.1100
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9860 - loss: 0.0464 - 

In [38]:
test_loss, test_acc = model.evaluate(X_test, y_test)

print(test_acc)
print(test_loss)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9723 - loss: 0.0886
0.9722999930381775
0.08863862603902817


In [39]:
# 예측
predictions = model.predict(X_test[:1])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step


In [42]:
print(predictions.argmax())

7
